# 1. Inicialización del Entorno Spark

**Dimensión 2: Utilización histórica de estaciones** — Clasificación de alta demanda.

Integrante: **Grimaldo Arredondo Martinez** — Data Geniuses

Se procesan los ~5 millones de viajes de Citi Bike NYC de forma distribuida.


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("U1_Utilizacion_Estaciones_CitiBike")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/11 14:19:18 WARN Utils: Your hostname, Jhack, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/11 14:19:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 14:19:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


# 2. Definición del Esquema Explícito y Carga de Datos (Capa Bronze)

Lectura raw con `StructType` explícito para tipado correcto de cada columna.

La ruta de datos se resuelve de forma portable: en el contenedor usa
`/opt/UNIDAD1/data/*.csv`; en ejecución local usa la ruta relativa `../data/*.csv`.


In [2]:
import os as _os

_has_opt = _os.path.exists("/opt/UNIDAD1")
_cwd = _os.getcwd()

DATA_PATH = "/opt/UNIDAD1/data/*.csv"
GOLD_ROOT = "/opt/UNIDAD1/gold"
if not _has_opt:
    DATA_PATH = _os.path.join(_cwd, "..", "data", "*.csv")
    GOLD_ROOT = _os.path.join(_cwd, "..", "gold")

from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, DoubleType,
)

schema_citibike = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True),
])

df = spark.read.schema(schema_citibike).option("header", True).csv(DATA_PATH)

26/09/11 14:19:22 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /home/jhackns/bigdata/Proyectos/grupo-x-citibike-intelligent-mobility/pyspark/jupyter/UNIDAD1/Dim-Estaciones/../data/*.csv.
java.io.FileNotFoundException: File /home/jhackns/bigdata/Proyectos/grupo-x-citibike-intelligent-mobility/pyspark/jupyter/UNIDAD1/data/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.cat

# 2.1 Verificación Inicial de Registros

Validamos la ingesta de los 5 archivos CSV.


In [3]:
total_registros = df.count()
print(f"Registros ingeridos: {total_registros:,}")

Registros ingeridos: 4,993,137


In [4]:
df.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



# 3. Limpieza Silver: Estaciones de Origen Válidas

Filtramos viajes con `start_station_name` no nulo, duración operativa
(1–180 minutos) y deduplicación por `ride_id`.


In [5]:
from pyspark.sql.functions import col, unix_timestamp

df = df.withColumn(
    "duration_minutes",
    (unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))) / 60.0,
)

In [6]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_clean = df.filter(
    col("start_station_name").isNotNull()
    & col("start_station_id").isNotNull()
    & col("duration_minutes").between(1.0, 180.0)
)

window_ride = Window.partitionBy("ride_id").orderBy(col("started_at").desc())
df_clean = (
    df_clean.withColumn("rn", row_number().over(window_ride))
            .filter(col("rn") == 1)
            .drop("rn")
)

n_validos = df_clean.count()
print(f"Viajes válidos: {n_validos:,} | Descartados: {total_registros - n_validos:,}")

Viajes válidos: 4,984,462 | Descartados: 8,675


In [7]:
df_clean.select(
    "ride_id", "start_station_id", "start_station_name",
    "start_lat", "start_lng", "started_at", "member_casual",
).show(5, truncate=False)

+----------------+----------------+-----------------------+------------------+------------------+-----------------------+-------------+
|ride_id         |start_station_id|start_station_name     |start_lat         |start_lng         |started_at             |member_casual|
+----------------+----------------+-----------------------+------------------+------------------+-----------------------+-------------+
|00002D5F1E4E52F5|6659.01         |E 51 St & Lexington Ave|40.75714758       |-73.97207836      |2026-07-15 18:49:42.373|casual       |
|00006E737685352B|5256.06         |Vesey St & West St     |40.714063         |-74.014432        |2026-07-16 21:04:17.572|member       |
|0000C3711CCB1498|4531.05         |Henry St & Atlantic Ave|40.69089272       |-73.99612349      |2026-07-25 20:29:04.34 |member       |
|0000E69756E4DC69|6416.06         |W 29 St & 9 Ave        |40.7500727        |-73.99839279      |2026-07-27 12:26:54.837|member       |
|00017E6FB27D30C3|6157.04         |10 Ave & W 14

# 4. Feature Engineering: Volumen por Estación

Agregamos los viajes por estación de origen (salidas) y de destino (llegadas)
para conformar el dataset de estudio a nivel de estación.


In [8]:
from pyspark.sql.functions import (
    count, countDistinct, avg, sum as spark_sum, col as ccol, to_date,
)

df_salidas = (
    df_clean.groupBy("start_station_id", "start_station_name")
            .agg(
                count("*").alias("salidas"),
                avg("start_lat").alias("lat_promedio"),
                avg("start_lng").alias("lng_promedio"),
                countDistinct(to_date("started_at")).alias("dias_activos"),
            )
)

df_llegadas = (
    df_clean.groupBy("end_station_id")
            .agg(count("*").alias("llegadas"))
            .withColumnRenamed("end_station_id", "estacion_id")
)

df_salidas = df_salidas.withColumnRenamed("start_station_id", "estacion_id")

In [9]:
df_estacion = (
    df_salidas
    .join(df_llegadas, on="estacion_id", how="left")
    .withColumn("llegadas", ccol("llegadas").cast("double"))
    .fillna({"llegadas": 0.0})
)

df_estacion = df_estacion.withColumn(
    "visitas_totales",
    ccol("salidas") + ccol("llegadas"),
)

#### 4.1 Viajes promedio por hora

Cada estación activa determinadas horas del día; calculamos la tasa de viajes
por hora activa como característica operativa.


In [10]:
from pyspark.sql.functions import hour

df_horas = (
    df_clean.groupBy("start_station_id")
            .agg(countDistinct(hour("started_at")).alias("horas_activas"))
            .withColumnRenamed("start_station_id", "estacion_id")
)

df_estacion = (
    df_estacion.join(df_horas, on="estacion_id", how="left")
    .withColumn("viajes_promedio_por_hora", ccol("salidas") / ccol("horas_activas"))
)

# 4.2 Verificación del Dataset de Estaciones

Número de estaciones, muestra y estadística descriptiva.


In [11]:
print(f"Total de estaciones analizadas: {df_estacion.count():,}")
df_estacion.orderBy(ccol("visitas_totales").desc()).show(10, truncate=False)

Total de estaciones analizadas: 2,305


+-----------+--------------------------+-------+------------------+------------------+------------+--------+---------------+-------------+------------------------+
|estacion_id|start_station_name        |salidas|lat_promedio      |lng_promedio      |dias_activos|llegadas|visitas_totales|horas_activas|viajes_promedio_por_hora|
+-----------+--------------------------+-------+------------------+------------------+------------+--------+---------------+-------------+------------------------+
|6233.04    |Pier 61 at Chelsea Piers  |18247  |40.74687199999999 |-74.00820999999998|32          |18268.0 |36515.0        |24           |760.2916666666666       |
|6140.05    |W 21 St & 6 Ave           |16724  |40.741739690000024|-73.99415556      |32          |16750.0 |33474.0        |24           |696.8333333333334       |
|5712.03    |Cooper Square & Astor Pl  |15717  |40.72951496224951 |-73.99075269699097|32          |15791.0 |31508.0        |24           |654.875                 |
|6492.08    |9 A

In [12]:
df_estacion.describe("salidas", "llegadas", "visitas_totales", "viajes_promedio_por_hora").show()

26/09/11 14:19:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+-----------------+------------------------+
|summary|           salidas|          llegadas|  visitas_totales|viajes_promedio_por_hora|
+-------+------------------+------------------+-----------------+------------------------+
|  count|              2305|              2305|             2305|                    2305|
|   mean| 2162.456399132321|2131.9631236442515|4294.419522776573|        90.5483238493853|
| stddev|2624.4920406236556|2594.6960751047213|5212.355667271376|      109.17140053515634|
|    min|                 1|               2.0|              3.0|                     1.0|
|    max|             18247|           18268.0|          36515.0|       760.2916666666666|
+-------+------------------+------------------+-----------------+------------------------+



# 5. Definición de la Clase Objetivo (Label): Alta Demanda

Umbral estadístico reproducible: **percentil 75** del volumen total de visitas.
`alta_demanda = 1` si la estación supera el umbral; `0` en caso contrario.


In [13]:
umbral_p75 = df_estacion.approxQuantile("visitas_totales", [0.75], 0.01)[0]
print(f"Umbral alta demanda (P75 de visitas): {umbral_p75:.2f}")

df_estacion = df_estacion.withColumn(
    "label",
    ccol("visitas_totales").cast("double") >= umbral_p75,
)
df_estacion = df_estacion.withColumn(
    "label",
    ccol("label").cast("double"),
)

Umbral alta demanda (P75 de visitas): 5854.00


In [14]:
print("Distribución de la clase objetivo:")
df_estacion.groupBy("label").count().orderBy("label").show()

Distribución de la clase objetivo:


+-----+-----+
|label|count|
+-----+-----+
|  0.0| 1706|
|  1.0|  599|
+-----+-----+



# 6. Pipeline ML: VectorAssembler y División Train/Test

Empaquetamos las variables de la estación en `features` y dividimos en
Train (80%) / Test (20%) con semilla reproducible.


In [15]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "lat_promedio", "lng_promedio",
    "viajes_promedio_por_hora",
    "dias_activos", "horas_activas",
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_model = assembler.transform(df_estacion)

train, test = df_model.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count():,} | Test: {test.count():,}")

Train: 1,901 | Test: 404


# 7. Entrenamiento de Modelos de Clasificación (Spark MLlib)

Se entrenan dos clasificadores distribuidos y se selecciona el de mejor AUC:
- **LogisticRegression** (referencia lineal)
- **RandomForestClassifier** (no lineal, robusto a outliers)


In [16]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
model_lr = lr.fit(train)

rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                            numTrees=50, maxDepth=6, seed=42)
model_rf = rf.fit(train)

preds_lr = model_lr.transform(test)
preds_rf = model_rf.transform(test)

auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc_lr = auc_eval.evaluate(preds_lr)
auc_rf = auc_eval.evaluate(preds_rf)
print(f"AUC LogisticRegression : {auc_lr:.4f}")
print(f"AUC RandomForest       : {auc_rf:.4f}")

best_model_name = "RandomForestClassifier" if auc_rf >= auc_lr else "LogisticRegression"
best_preds = preds_rf if auc_rf >= auc_lr else preds_lr
print(f">> Modelo seleccionado por AUC: {best_model_name}")

AUC LogisticRegression : 0.9988
AUC RandomForest       : 0.9987
>> Modelo seleccionado por AUC: LogisticRegression


# 8. Evaluación Formal del Modelo de Clasificación

Métricas: **Accuracy, F1, Precision, Recall** (MulticlassClassificationEvaluator)
y **Matriz de Confusión** sobre el conjunto de test.


In [17]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(best_preds)
f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(best_preds)
prec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision").evaluate(best_preds)
rec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall").evaluate(best_preds)

In [18]:
print("=" * 52)
print("MÉTRICAS DE CLASIFICACIÓN (UTILIZACIÓN DE ESTACIONES)")
print("=" * 52)
print(f"  Accuracy  : {acc:10.4f}")
print(f"  Precision : {prec:10.4f}")
print(f"  Recall    : {rec:10.4f}")
print(f"  F1-Score  : {f1:10.4f}")
print(f"  AUC-ROC   : {auc_rf if best_model_name == 'RandomForestClassifier' else auc_lr:10.4f}")
print("=" * 52)

MÉTRICAS DE CLASIFICACIÓN (UTILIZACIÓN DE ESTACIONES)
  Accuracy  :     0.9851
  Precision :     0.9852
  Recall    :     0.9851
  F1-Score  :     0.9851
  AUC-ROC   :     0.9988


In [19]:
print("Matriz de Confusión (Test):")
best_preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

Matriz de Confusión (Test):


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  297|
|  0.0|       1.0|    1|
|  1.0|       0.0|    5|
|  1.0|       1.0|  101|
+-----+----------+-----+



# 9. Ranking de Estaciones y Persistencia en Capa Gold

Generamos el ranking de estaciones por `visitas_totales` y persistimos el
artefacto analítico en **Parquet** con el efecto de predicción.


In [20]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, round as spark_round
from pyspark.ml.functions import vector_to_array

window_rank = Window.orderBy(ccol("visitas_totales").desc())
df_ranking = df_estacion.withColumn("ranking", row_number().over(window_rank))

df_gold = (
    best_preds.join(df_ranking.select("estacion_id", "ranking"), on="estacion_id", how="left")
    .select(
        "estacion_id",
        ccol("start_station_name").alias("estacion_nombre"),
        "salidas", "llegadas",
        "visitas_totales", "label", "prediction",
        spark_round(vector_to_array(ccol("probability"))[1], 4).alias("prob_alta_demanda"),
        "ranking",
    )
    .orderBy("ranking")
)

GOLD_PATH = GOLD_ROOT + "/estaciones_predictions.parquet"
df_gold.write.mode("overwrite").parquet(GOLD_PATH)
print(f"Artefacto persistido: {GOLD_PATH}")

26/09/11 14:23:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/11 14:23:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/11 14:23:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/11 14:23:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/11 14:23:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/11 14:23:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/11 14:23:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/11 14:23:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/11 14:23:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/09/11 14:23:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 14:23:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 1

Artefacto persistido: /home/jhackns/bigdata/Proyectos/grupo-x-citibike-intelligent-mobility/pyspark/jupyter/UNIDAD1/Dim-Estaciones/../gold/estaciones_predictions.parquet


In [21]:
gold_check = spark.read.parquet(GOLD_PATH)
print(f"Estaciones en capa Gold: {gold_check.count():,}")
gold_check.show(10, truncate=False)

Estaciones en capa Gold: 406
+-----------+-----------------------+-------+--------+---------------+-----+----------+-----------------+-------+
|estacion_id|estacion_nombre        |salidas|llegadas|visitas_totales|label|prediction|prob_alta_demanda|ranking|
+-----------+-----------------------+-------+--------+---------------+-----+----------+-----------------+-------+
|6492.08    |9 Ave & W 33 St        |14838  |14888.0 |29726.0        |1.0  |1.0       |1.0              |4      |
|5905.14    |University Pl & E 14 St|13399  |13447.0 |26846.0        |1.0  |1.0       |1.0              |11     |
|6157.04    |10 Ave & W 14 St       |13018  |13076.0 |26094.0        |1.0  |1.0       |1.0              |13     |
|6257.03    |W 24 St & 7 Ave        |12065  |12068.0 |24133.0        |1.0  |1.0       |1.0              |21     |
|5340.01    |N 7 St & Driggs Ave    |11408  |11305.0 |22713.0        |1.0  |1.0       |1.0              |27     |
|6289.06    |Broadway & W 29 St     |11119  |11131.0 |22250